# 02 - Calendar-only baselines

Create baseline models with calendar data only.

Period day 1 is the first reported day of flow, excluding spotting. For Interval 2, which has no
flow reports, it is the first day of a run of `phase == 'Menstrual'`.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data, baselines, evaluation

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
FIGDIR = os.path.join("..", "outputs", "figures")
RESDIR = os.path.join("..", "outputs", "results")
SEED = 0

## Labels

Both intervals, pooled. Interval 2 labels come from `phase`, so they are a supplied label mixed in
with self-reports.

In [ ]:
diary  = data.load_diary()
starts = data.period_starts(diary, gap_days=3)
cycles = data.build_cycles(starts, min_len=15, max_len=60)
frame  = data.make_daily_prediction_frame(cycles)

print(f"{len(cycles)} cycles from {cycles.id.nunique()} participants, {len(frame):,} prediction rows")
print(cycles.groupby("study_interval").size().to_string())
data.censoring_report(starts, cycles)

## The three models

Days remaining is `cycle_length - cycle_day + 1`, so predicting it means estimating the current
cycle's length.

| model | estimate | source |
|---|---|---|
| population | one cycle length for everyone | training participants' cycles |
| personalised | that participant's own median | their earlier cycles only |
| calendar-day | no assumed length | average days remaining at each cycle day |

## Splitting

In [ ]:
split = data.participant_split(cycles, test_frac=0.3, seed=SEED)
train_cycles, test_cycles = split.apply(cycles)
train_frame,  test_frame  = split.apply(frame)

print(f"train: {len(split.train_ids)} participants, {len(train_cycles)} cycles, {len(train_frame):,} days")
print(f"test:  {len(split.test_ids)} participants, {len(test_cycles)} cycles, {len(test_frame):,} days")
print(f"overlap: {len(set(split.train_ids) & set(split.test_ids))}")

## Fitting

The personalised model is given the test participants' cycles but reads only cycles earlier than
the one being predicted, which matches knowing a returning user's history but not their current
cycle.

In [ ]:
population = baselines.PopulationBaseline(statistic="median").fit(train_cycles)
calendar   = baselines.CalendarDayModel().fit(train_frame)
personal   = baselines.PersonalBaseline(statistic="median", fallback=population).fit(test_cycles)

print(f"population cycle length, from training only: {population.cycle_length_:.1f} days")

In [ ]:
ev = test_frame.copy()
ev["pred_population"] = population.predict(ev)
ev["pred_calendar"]   = calendar.predict(ev)
ev["pred_personal"], used_fallback = personal.predict(ev, return_fallback_mask=True)

print(f"rows where the personalised model had no earlier cycle: {100 * used_fallback.mean():.0f}%")

On those rows the personalised model returns the population estimate, so part of the
personalised column is the population model. The fallback is fitted on training participants only,
so no test data reaches it.

## New participants against returning ones

In [ ]:
cold = ev[ev.cycle_index == 0]

seen = frame[frame.cycle_index >= 1].copy()
seen["pred_personal"], _ = baselines.predict_expanding_personal(seen, cycles, fallback=population)
seen["pred_population"] = population.predict(seen)

regimes = pd.DataFrame([
    {"regime": "new participant, cycle 0", "n_participants": cold.id.nunique(), "n_days": len(cold),
     "personal_MAE": round(evaluation.mae(cold.days_until_next_period, cold.pred_personal), 3),
     "population_MAE": round(evaluation.mae(cold.days_until_next_period, cold.pred_population), 3)},
    {"regime": "returning, cycle 1+", "n_participants": seen.id.nunique(), "n_days": len(seen),
     "personal_MAE": round(evaluation.mae(seen.days_until_next_period, seen.pred_personal), 3),
     "population_MAE": round(evaluation.mae(seen.days_until_next_period, seen.pred_population), 3)},
])
regimes

## Results across many splits

Refit on 20 random participant splits.

In [ ]:
def fit_predict(train_cycles, train_frame, test_cycles, test_frame):
    pop = baselines.PopulationBaseline().fit(train_cycles)
    cal = baselines.CalendarDayModel().fit(train_frame)
    per = baselines.PersonalBaseline(fallback=pop).fit(test_cycles)
    return {"population": pop.predict(test_frame),
            "personalised": per.predict(test_frame),
            "calendar-day": cal.predict(test_frame)}

scores = evaluation.repeated_split_scores(cycles, frame, fit_predict, n_splits=20)
split_summary = evaluation.summarise_splits(scores)
split_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
order = split_summary.model.tolist()
ax.boxplot([scores.loc[scores.model == m, "MAE"] for m in order], labels=order, vert=True, widths=0.5)
for x, m in enumerate(order, start=1):
    ax.scatter(np.full(20, x) + np.random.default_rng(0).normal(0, 0.04, 20),
               scores.loc[scores.model == m, "MAE"], s=14, alpha=0.6, color="#2b6cb0")
ax.set_ylabel("MAE (days)")
ax.set_title("Each dot is one participant split", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "split_spread.png"), dpi=150, bbox_inches="tight")

None of the models is measurably better than the others on this data.

Choose one split as a set of predictions for the plots.

In [ ]:
preds = {"population": "pred_population", "personalised": "pred_personal",
         "calendar-day": "pred_calendar"}
results = evaluation.compare_models(ev, "days_until_next_period", preds, n_boot=2000, seed=SEED)
results

## Predicted against actual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (label, col) in zip(axes, preds.items()):
    evaluation.plot_pred_vs_actual(ax, ev.days_until_next_period, ev[col], label, max_day=45)
fig.suptitle("Predicted against actual days until next period, held-out participants", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "pred_vs_actual.png"), dpi=150, bbox_inches="tight")

The population and personalised models trace diagonal bands, one per assumed cycle length,
because they subtract the cycle day from a constant. The calendar-day model has a floor where it
falls back to the average.

## Where the error sits

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
evaluation.plot_mae_by_cycle_day(ax, ev, "days_until_next_period", preds, max_day=35)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "mae_by_cycle_day.png"), dpi=150, bbox_inches="tight")

Error is worst early in the cycle, where the horizon is longest, and falls as the next period
approaches. Early cycle is where a temperature or resting heart rate signal has the most room to
help.

## Per participant

In [ ]:
pp = evaluation.per_participant_mae(ev, "days_until_next_period", "pred_population")
summary = pp.MAE.describe()[["count", "mean", "50%", "min", "max"]]
summary.index = ["n_participants", "mean", "median", "best", "worst"]
summary.round(2).to_frame("per_participant_MAE").T

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
for label, col in preds.items():
    s = evaluation.per_participant_mae(ev, "days_until_next_period", col).MAE.sort_values().values
    ax.plot(range(1, len(s) + 1), s, "-o", ms=4, label=label)
ax.set_xlabel("held-out participant, sorted by error")
ax.set_ylabel("MAE (days)")
ax.set_title("Participants differ more than the models do", fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "per_participant_mae.png"), dpi=150, bbox_inches="tight")

The spread across participants is wider than the gap between models.

## Save results

In [ ]:
os.makedirs(RESDIR, exist_ok=True)
payload = {
    "labelling": {"gap_days": 3, "min_cycle_len": 15, "max_cycle_len": 60,
                  "cycles": int(len(cycles)), "participants": int(cycles.id.nunique()),
                  "cycles_by_interval": {str(k): int(v) for k, v in cycles.groupby("study_interval").size().items()}},
    "split": {"train_participants": int(len(split.train_ids)),
              "test_participants": int(len(split.test_ids)), "seed": SEED},
    "population_cycle_length_days": round(float(population.cycle_length_), 2),
    "personal_fallback_rate_pct": round(100 * float(used_fallback.mean()), 1),
    "repeated_splits": json.loads(split_summary.to_json(orient="records")),
    "single_split": json.loads(results.to_json(orient="records")),
    "regimes": json.loads(regimes.to_json(orient="records")),
}
with open(os.path.join(RESDIR, "baseline_results.json"), "w") as fh:
    json.dump(payload, fh, indent=2)

with open(os.path.join(RESDIR, "baseline_results.md"), "w") as fh:
    fh.write("# Step 2, calendar-only baselines\n\n")
    fh.write(f"{len(cycles)} cycles from {cycles.id.nunique()} participants, 20 random splits, gap_days=3.\n\n")
    fh.write(split_summary.to_markdown(index=False))
    fh.write("\n\n## New against returning participants\n\n")
    fh.write(regimes.to_markdown(index=False) + "\n")
print("wrote outputs/results/baseline_results.{json,md}")

## Notes

1. Across 20 splits the three models are within about a fifth of a day of each other, while a
   single split moves the answer by more than three days. This data cannot rank them.
2. The personalised model does not beat the population one. At a median of three cycles per
   participant, its estimate is often a single earlier cycle, which is noisier than a cohort median
   built from many people. About half of held-out days have no earlier cycle at all.
3. Differences between participants are larger than differences between models.
4. Error concentrates early in the cycle.
5. Pooling Interval 2 helps: 119 cycles gives a lower mean MAE than Interval 1's 76 on its own.
6. Spotting stays excluded. Counting it raises MAE and drops three participants below two cycles.

Conditional on gap_days=3 and the 15 to 60 day bounds.

For Step 3, the bar is the mean across splits, not a single number, and a wearable model has to
clear a gap wider than the models here differ from each other.